In [ ]:
# Ноутбук перенесён в подпапку — восстанавливаем CWD в корень thesis/
import os
from pathlib import Path

if Path.cwd().name != "thesis":
    os.chdir("..")
print("CWD:", Path.cwd())


# Создание LanceDB базы и векторизация постов

Этот ноутбук:
1. Загружает очищенные посты из `selected500k_cleaned.jsonl`
2. Кодирует их дообученным bi-encoder (`models/final/bi-encoder`)
3. Записывает в LanceDB (персистентная база на диске)
4. Создаёт FTS-индекс для полнотекстового поиска (BM25)

**Поддерживает остановку и возобновление** — при перезапуске продолжит с того места, где остановился.

In [7]:
# Установка зависимостей (один раз)
# !pip install lancedb

In [8]:
# ==================== НАСТРОЙКИ ====================

# Пути
POSTS_FILE = "data/posts/ajtkulov/selected/selected500k_cleaned.jsonl"
BI_ENCODER_PATH = "models/final/bi-encoder"
LANCEDB_PATH = "./lancedb_store"          # папка для базы данных
TABLE_NAME = "rosberta-fine-tuned-50k"                       # имя таблицы в LanceDB

# Параметры загрузки
MAX_POSTS = 50_000                         # сколько постов загрузить (None = все)
BATCH_SIZE = 512                           # размер батча для encode
DEVICE = "cuda"                            # "cuda" или "cpu"

# Категории: берём только основную (до |||)
CLEAN_CATEGORY = True

## 1. Загрузка модели

In [9]:
import warnings
warnings.filterwarnings('ignore')

import torch

# Совместимость sentence-transformers 5.x + transformers 4.57
import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer

device = DEVICE if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

bi_encoder = SentenceTransformer(BI_ENCODER_PATH, device=device)
EMBEDDING_DIM = bi_encoder.get_sentence_embedding_dimension()
print(f"\nМодель загружена: {BI_ENCODER_PATH}")
print(f"  Embedding dim: {EMBEDDING_DIM}")
print(f"  Max seq len: {bi_encoder.max_seq_length}")

Device: cpu

Модель загружена: models/final/bi-encoder
  Embedding dim: 384
  Max seq len: 8192


## 2. Загрузка постов

In [10]:
import json
import random
from tqdm.auto import tqdm

RANDOM_SEED = 42  # для воспроизводимости

def load_posts(path, max_posts=None, shuffle=True, seed=RANDOM_SEED):
    """Загружает посты из JSONL. Если max_posts задан — выбирает случайные."""
    all_posts = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Чтение постов"):
            obj = json.loads(line)
            # Пропускаем посты с пустым текстом
            if not obj.get('text', '').strip():
                continue
            all_posts.append(obj)
    
    print(f"Всего постов в файле: {len(all_posts):,}")
    
    if max_posts and max_posts < len(all_posts):
        rng = random.Random(seed)
        all_posts = rng.sample(all_posts, max_posts)
        print(f"Случайная выборка: {len(all_posts):,} (seed={seed})")
    
    return all_posts

posts = load_posts(POSTS_FILE, max_posts=MAX_POSTS)
print(f"\nЗагружено постов: {len(posts):,}")
print(f"Пример: {posts[0]['channel']} — {posts[0]['text'][:100]}...")

Чтение постов: 0it [00:00, ?it/s]

Всего постов в файле: 499,855
Случайная выборка: 50,000 (seed=42)

Загружено постов: 50,000
Пример: obshakstaya — Ма Тут активация пишет недействительны, значит вы просто не успели) Все разобрали, в некст раз повез...


In [11]:
# Очистка категорий: берём только основную часть (до |||)
if CLEAN_CATEGORY:
    for p in posts:
        cat = p.get('category', '')
        if '|||' in cat:
            p['category'] = cat.split('|||')[0].strip()

# Статистика
from collections import Counter
categories = Counter(p['category'] for p in posts)
channels = set(p['channel'] for p in posts)
print(f"Уникальных каналов: {len(channels):,}")
print(f"Уникальных категорий: {len(categories)}")
print(f"\nТоп-10 категорий:")
for cat, cnt in categories.most_common(10):
    print(f"  {cat}: {cnt:,}")

Уникальных каналов: 11,515
Уникальных категорий: 39

Топ-10 категорий:
  Блоги: 8,933
  Криптовалюты: 3,385
  Игры: 3,300
  Мода и красота: 3,101
  Юмор и развлечение: 2,067
  Продажи: 1,928
  Технологии: 1,871
  Экономика: 1,844
  Психология: 1,723
  Еда и кулинария: 1,572


## 3. Подключение к LanceDB и проверка прогресса

In [12]:
import lancedb
import pyarrow as pa
import numpy as np

db = lancedb.connect(LANCEDB_PATH)

# Схема таблицы
schema = pa.schema([
    pa.field("vector", pa.list_(pa.float32(), EMBEDDING_DIM)),
    pa.field("text", pa.utf8()),
    pa.field("channel", pa.utf8()),
    pa.field("category", pa.utf8()),
    pa.field("post_id", pa.utf8()),       # уникальный ID: channel + id поста
    pa.field("link", pa.utf8()),
    pa.field("date", pa.utf8()),
    pa.field("views", pa.utf8()),
])

# Проверяем, есть ли уже таблица (для возобновления)
existing_tables = db.table_names()
already_indexed = 0
indexed_ids = set()

if TABLE_NAME in existing_tables:
    table = db.open_table(TABLE_NAME)
    already_indexed = table.count_rows()
    print(f"Таблица '{TABLE_NAME}' уже существует: {already_indexed:,} записей")
    
    # Загружаем ID уже проиндексированных постов
    if already_indexed > 0:
        existing_df = table.to_pandas(columns=["post_id"])
        indexed_ids = set(existing_df["post_id"].tolist())
        print(f"Загружено {len(indexed_ids):,} уникальных ID для проверки дубликатов")
else:
    table = db.create_table(TABLE_NAME, schema=schema)
    print(f"Создана новая таблица '{TABLE_NAME}'")

Создана новая таблица 'posts'


In [13]:
# Фильтруем посты — оставляем только те, которых ещё нет в базе
def make_post_id(post):
    """Уникальный ID поста: channel::id"""
    return f"{post['channel']}::{post['id']}"

if indexed_ids:
    posts_to_index = [p for p in posts if make_post_id(p) not in indexed_ids]
    print(f"Пропущено (уже в базе): {len(posts) - len(posts_to_index):,}")
    print(f"Осталось проиндексировать: {len(posts_to_index):,}")
else:
    posts_to_index = posts
    print(f"Индексируем всё: {len(posts_to_index):,} постов")

Индексируем всё: 50,000 постов


## 4. Векторизация и запись в LanceDB

Записываем батчами. Прогресс сохраняется автоматически — при прерывании (Ctrl+C / остановка ядра) уже записанные батчи останутся в базе. При повторном запуске продолжит с того места, где остановился.

In [14]:
import time

total = len(posts_to_index)
if total == 0:
    print("Все посты уже в базе — нечего индексировать!")
else:
    print(f"Старт: {total:,} постов, батч {BATCH_SIZE}, device={device}")
    print(f"Оценка: ~{total / (300 if device == 'cuda' else 30):.0f} секунд")
    print()
    
    t_start = time.time()
    indexed_count = 0
    
    pbar = tqdm(total=total, desc="Векторизация", unit="пост")
    
    try:
        for batch_start in range(0, total, BATCH_SIZE):
            batch_posts = posts_to_index[batch_start : batch_start + BATCH_SIZE]
            texts = [p['text'] for p in batch_posts]
            
            # Encode батч
            embeddings = bi_encoder.encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=False,
                batch_size=BATCH_SIZE,
                device=device
            )
            
            # Формируем записи для LanceDB
            records = []
            for i, post in enumerate(batch_posts):
                records.append({
                    "vector": embeddings[i].tolist(),
                    "text": post['text'],
                    "channel": post['channel'],
                    "category": post.get('category', ''),
                    "post_id": make_post_id(post),
                    "link": post.get('link', ''),
                    "date": post.get('date', ''),
                    "views": str(post.get('views', '')),
                })
            
            # Записываем в базу (атомарно — либо весь батч, либо ничего)
            table.add(records)
            indexed_count += len(records)
            
            # Обновляем прогресс
            pbar.update(len(records))
            elapsed = time.time() - t_start
            speed = indexed_count / elapsed
            eta = (total - indexed_count) / speed if speed > 0 else 0
            pbar.set_postfix({
                "скорость": f"{speed:.0f} п/с",
                "ETA": f"{eta/60:.1f} мин",
                "в базе": f"{already_indexed + indexed_count:,}"
            })
    
    except KeyboardInterrupt:
        print(f"\n\nОстановлено пользователем!")
        print(f"Записано в этом запуске: {indexed_count:,}")
        print(f"Всего в базе: {already_indexed + indexed_count:,}")
        print(f"Для продолжения — просто перезапустите ноутбук с ячейки 3.")
    
    else:
        elapsed = time.time() - t_start
        print(f"\nГотово!")
        print(f"Записано: {indexed_count:,} постов за {elapsed:.0f}с ({indexed_count/elapsed:.0f} п/с)")
        print(f"Всего в базе: {already_indexed + indexed_count:,}")
    
    finally:
        pbar.close()

Старт: 50,000 постов, батч 512, device=cpu
Оценка: ~1667 секунд



Векторизация:   0%|          | 0/50000 [00:00<?, ?пост/s]


Готово!
Записано: 50,000 постов за 4196с (12 п/с)
Всего в базе: 50,000


## 5. Создание FTS-индекса (для полнотекстового BM25-поиска)

In [15]:
# FTS-индекс нужно пересоздавать после добавления данных
# (replace=True — пересоздаёт, если уже был)

print("Создание FTS-индекса на поле 'text'...")
table.create_fts_index("text", replace=True)
print("Готово!")

Создание FTS-индекса на поле 'text'...
Готово!


## 6. Проверка: три режима поиска

In [16]:
# Тестовый запрос — товарное описание
test_query = "Кроссовки Nike Air Max мужские для бега, размер 42, чёрные"

print(f"Запрос: {test_query}")
print("=" * 80)

Запрос: Кроссовки Nike Air Max мужские для бега, размер 42, чёрные


In [17]:
# --- 6а. Векторный поиск ---
print("\n🔍 ВЕКТОРНЫЙ ПОИСК (bi-encoder)")
print("-" * 40)

query_vec = bi_encoder.encode([test_query], normalize_embeddings=True)[0].tolist()

results_vec = (
    table.search(query_vec, query_type="vector")
    .limit(5)
    .select(["text", "channel", "category"])
    .to_list()
)

for i, r in enumerate(results_vec, 1):
    print(f"  {i}. [{r['category']}] @{r['channel']}")
    print(f"     score={r['_distance']:.4f}")
    print(f"     {r['text'][:120]}...")
    print()


🔍 ВЕКТОРНЫЙ ПОИСК (bi-encoder)
----------------------------------------
  1. [Мода и красота] @drip_wb
     score=0.1023
     #обувьКроссовки Adidas Human Made x Adimatic - 3276₽Кожаные кроссовки на платформе дышащие с сеткой и ушками - 2427₽Крос...

  2. [Мода и красота] @drip_wb
     score=0.1023
     #обувьКроссовки Adidas Human Made x Adimatic - 3276₽Кожаные кроссовки на платформе дышащие с сеткой и ушками - 2427₽Крос...

  3. [Продажи] @darom_ali
     score=0.1178
     Тёплые кроссовки 555р штаны 201р свитер 242р 242р лезвия для Gillette Fusion 296р 178р...

  4. [Продажи] @dmitry_soldatov
     score=0.1219
     Мужские кроссовкиЦена: 1500Размеры: 41-45Качество: ЛюксДля заказов WhatsApp заказа пришлите размер и фотографию модели и...

  5. [Продажи] @dmitry_soldatov
     score=0.1219
     Мужские кроссовкиЦена: 1500Размеры: 41-45Качество: ЛюксДля заказов WhatsApp заказа пришлите размер и фотографию модели и...



In [18]:
# --- 6б. Полнотекстовый поиск (BM25) ---
print("📝 ПОЛНОТЕКСТОВЫЙ ПОИСК (BM25)")
print("-" * 40)

results_fts = (
    table.search(test_query, query_type="fts")
    .limit(5)
    .select(["text", "channel", "category"])
    .to_list()
)

for i, r in enumerate(results_fts, 1):
    print(f"  {i}. [{r['category']}] @{r['channel']}")
    print(f"     score={r.get('_score', r.get('_relevance_score', 'N/A'))}")
    print(f"     {r['text'][:120]}...")
    print()

📝 ПОЛНОТЕКСТОВЫЙ ПОИСК (BM25)
----------------------------------------
  1. [Мода и красота] @soda_fashion_store
     score=29.126678466796875
     Кроссовки Nike Off-White x Air Rubber Dunk 'Green Strike' Размеры: 36-45 Цена: 5790₽Навигация по магазину Для заказа пиш...

  2. [Мода и красота] @poizonqq
     score=24.648303985595703
     Nike Sportswear представили очередную расцветку для своего силуэта Air Max Scorpion, которая войдет в коллекцию «Leap Hi...

  3. [Мода и красота] @poizonqq
     score=24.648303985595703
     Nike Sportswear представили очередную расцветку для своего силуэта Air Max Scorpion, которая войдет в коллекцию «Leap Hi...

  4. [Мода и красота] @Clotheshunter
     score=22.20940399169922
     Подборка крос на лето, старался выбирать лёгкие и светлые.1. Кеды Diadora | sale 3 843 ₽ КРОССОВКИ REEBOK ROYAL TECHQUE ...

  5. [Мода и красота] @pablo_msk_store
     score=22.047414779663086
     Nike Air Force 1 Gore TexРазмеры в наличии:44 / 10 US / 9 UK / 28 Цена: 1

In [19]:
# --- 6в. Гибридный поиск (vector + BM25) ---
from lancedb.rerankers import LinearCombinationReranker

print("🔀 ГИБРИДНЫЙ ПОИСК (vector + BM25, weight=0.7)")
print("-" * 40)

reranker = LinearCombinationReranker(weight=0.7)

results_hybrid = (
    table.search(query_type="hybrid")
    .vector(query_vec)
    .text(test_query)
    .limit(5)
    .rerank(reranker=reranker)
    .select(["text", "channel", "category"])
    .to_list()
)

for i, r in enumerate(results_hybrid, 1):
    print(f"  {i}. [{r['category']}] @{r['channel']}")
    print(f"     score={r.get('_relevance_score', 'N/A')}")
    print(f"     {r['text'][:120]}...")
    print()

🔀 ГИБРИДНЫЙ ПОИСК (vector + BM25, weight=0.7)
----------------------------------------
  1. [Мода и красота] @pablo_msk_store
     score=1.0
     Nike Air Force 1 Gore TexРазмеры в наличии:44 / 10 US / 9 UK / 28 Цена: 12 000Состояние: Новый с родным боксомДоп фото /...

  2. [Мода и красота] @Clotheshunter
     score=0.9931353330612183
     Подборка крос на лето, старался выбирать лёгкие и светлые.1. Кеды Diadora | sale 3 843 ₽ КРОССОВКИ REEBOK ROYAL TECHQUE ...

  3. [Мода и красота] @poizonqq
     score=0.8897813558578491
     Nike Sportswear представили очередную расцветку для своего силуэта Air Max Scorpion, которая войдет в коллекцию «Leap Hi...

  4. [Мода и красота] @poizonqq
     score=0.8897813558578491
     Nike Sportswear представили очередную расцветку для своего силуэта Air Max Scorpion, которая войдет в коллекцию «Leap Hi...

  5. [Продажи] @dmitry_soldatov
     score=0.699999988079071
     Мужские кроссовкиЦена: 1500Размеры: 41-45Качество: ЛюксДля заказов WhatsApp заказа

## 7. Статистика базы

In [20]:
import os

# Размер на диске
def dir_size_mb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 * 1024)

total_rows = table.count_rows()
db_size = dir_size_mb(LANCEDB_PATH)

print(f"Записей в базе:  {total_rows:,}")
print(f"Размер на диске: {db_size:.1f} MB")
print(f"Среднее на запись: {db_size * 1024 / total_rows:.1f} KB")
print(f"\nОценка для 500k: ~{db_size * 500_000 / total_rows:.0f} MB")

Записей в базе:  50,000
Размер на диске: 122.0 MB
Среднее на запись: 2.5 KB

Оценка для 500k: ~1220 MB


## Как возобновить

Если вы остановили процесс (Ctrl+C или остановка ядра):

1. Перезапустите ядро
2. Запустите ячейки с **1 по 4** — скрипт сам определит, какие посты уже в базе, и продолжит с оставшихся

Для полной переиндексации — удалите папку `lancedb_store/` и запустите заново.

Чтобы увеличить лимит до всех 500k, измените `MAX_POSTS = None` в настройках.